# Tutorial 04: Deep Dive: Exploring Your Results

**About this tutorial**
- **Time:** ~30 minutes
- **Prerequisites:** [Tutorial 03](03_core_phenotyping_workflow.ipynb) completed or understood
- **Key ideas:** Clustering, prototypes, interpretability, linking embeddings to classical properties

**Before you start:** Run this from `Notebooks/tutorials/`. If you are new to the pipeline, skim [03_core_phenotyping_workflow.ipynb](03_core_phenotyping_workflow.ipynb) first.

**In plain terms:** The vision model turns each image into an **embedding** (a numeric summary of appearance). This notebook shows typical **next steps** for digging deeper: grouping similar cells and finding patterns (clustering), identifying the most "typical" cells (prototypes), and linking the AI's complex findings back to real biological traits (explainability).

For a unified **HTML report** that bundles many of these sections automatically, use `generate_report()` and see the [Report API docs](../../docs/src/content/docs/advanced/api/report.md).

**Author:** Aitor González-Marfil ([@AAitorG](https://github.com/AAitorG))

---

## 1. Setup

In [ ]:
import torch
from phenome import PhenoMe, get_metadata_from_path, load_dinov2_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
wrapper = load_dinov2_model(model_name="dinov2_vitb14_reg", device=device)
pheno = PhenoMe(device=device, seed=42)

## 2. Load data, embed, optional properties

Same pattern as the other examples: discover files, run `process_images`, then optionally `compute_properties` if you linked **masks** in `find_files` (required for presets like `basic`). Adjust `reference_filters` so the **control** label matches your metadata (here: `condition == "Control"`).

In [ ]:
metadata_fn = get_metadata_from_path(".../(condition)/(filename).*")
image_dir = "path/to/images"
mask_dir = "path/to/masks" # optional, can be None if you don't have masks or if the directory doesn't exist

file_df = pheno.find_files(
    image_dir,
    metadata_fn=metadata_fn,
    mask_dir=mask_dir,
)
print(f"Found {len(file_df)} images")

pheno.process_images(wrapper, channel_mode="split")
pheno.compute_properties(property_preset="basic")

In [ ]:
# Plot number of images per group
pheno.plot_counts(group_by=["condition"])

## 3. Summaries by Experimental Group

This step checks if your physical measurements (like overall area or brightness) actually differ between your experimental groups (like 'Treated' vs. 'Untreated'). It requires that you have already run `compute_properties`.

In [ ]:
prop_keys = pheno.get_available_property_keys()

df_grouped = pheno.property_stats_by_group(
    group_by=["condition"],
    print_properties=prop_keys[:3],  # first 3 properties for brevity
)
# Grouped statistics table is printed when print_table=True (default).

### 3.1.1 Show top features different from reference

Shows which morphological features differ most between each group and a reference (e.g. control). This uses **Cohen's d** (effect size) by default to rank properties by how significant the difference is, regardless of the property's units.

*   **Cohen's d > 0.8** is generally considered a "large" effect.
*   **Rank by |d|** ensures we find the features that change most in either direction (larger or smaller).

Adjust `reference_group` to match your metadata.

In [ ]:
pheno.top_properties_different_from_reference(
    df_grouped,
    reference_group={"condition": "Control"},
    k=3,
)

## 4. Cluster in embedding space

Clusters are **data-driven** groups in embedding space; they may or may not match your experimental labels. Set `n_clusters` between 2 and the number of images (see [Pipeline API](../../docs/src/content/docs/advanced/api/pipeline.md)). After clustering, labels are stored under metadata key `cluster`.

In [ ]:
pheno.compute_clustering(n_clusters=5, clustering_method="kmeans")
pheno.plot_pca(color_by="cluster")

### 4.1. Group enrichment

Finding the defining traits of a group. Are cells in Cluster 1 significantly larger than everyone else? This helps you figure out exactly what biological features make a specific cluster stand out.

In [ ]:
# plot=False: logger text summary; set plot=True for an interactive faceted bar chart (optional: top_k, return_fig).
enrichment_df = pheno.analyze_group_enrichment(group_by="cluster", plot=False)


print("\nGroup enrichment results:")
for grp in sorted(enrichment_df["cluster"].unique())[:5]:
    sub = enrichment_df[enrichment_df["cluster"] == grp].head(5)
    print(f"\nGroup {grp} (top properties by |z-score|):")
    print(sub[["Property", "Score"]].to_string(index=False))

## 5. Representative / Prototype Images

When analyzing a large dataset, you often want to know what a typical cell in a specific cluster or condition looks like. **Prototypes** answer this by finding the most 'average' or 'typical' images for each group.

**How it works:** PhenoMe calculates the center (centroid) of each group in the high-dimensional embedding space, and finds the real images that are physically closest to that ideal center.

This is extremely useful when a clustering algorithm discovers a new group and you want to visually inspect what that phenotype represents.

In [ ]:
prototypes = pheno.find_prototypes(
    cluster_col="cluster", # Can also be 'condition', 'plate', etc.
    n_prototypes=3         # How many representative images to pull per group
)

for name, idxs in prototypes.items():
    print(f"Cluster {name} prototypes: {idxs}")
    # Plot the very best prototype for this group
    pheno.plot_image_by_index(idxs[0], show_extra_info=False)

## 6. Detecting Outliers

Just as prototypes find the most typical images, **outlier detection** finds the most highly unusual images within a group.

**Why detect outliers?**
1. **Quality Control (QC):** Find blurry, out-of-focus images, debris, or overlapping cells that shouldn't be analyzed.
2. **Rare Biological Events:** Find cells undergoing rare processes (like mitosis) that are valid, but very different from the norm.

**How it works:** PhenoMe measures the distance of every image to the center of its group. Images that are further away than a certain standard deviation `threshold` (e.g., 3.0) are flagged.

In [ ]:
outliers = pheno.detect_outliers(
    group_by="condition", # We expect cells within the same condition to look similar
    threshold=3.0         # Z-score distance threshold
)

if outliers["outlier_indices"]:
    print(f"Found {len(outliers['outlier_indices'])} outliers across all groups.")
    top_idx = outliers["outlier_indices"][0]
    print(f"Plotting the most extreme outlier (index: {top_idx})...")
    pheno.plot_image_by_index(top_idx, show_extra_info=False)
else:
    print("No outliers detected with current settings.")

## 7. Three Kinds of Explainability

When trying to understand what the AI is actually seeing, there are three main approaches:
- **1) Global explainers (`compute_embedding_property_correlations`):** Addresses *"Which physical traits line up with the model’s representation across embedding dimensions overall?"* Summarizes correlation across all dimensions.
- **2) Plot explainers (`compute_component_correlation`):** Addresses *"Why is this image on the left side of my plot?"* It ties the axes of your visual map (like PCA/UMAP) directly to real features like cell size using univariate correlation.
- **3) Multivariate explainers (`compute_multivariate_interpretability`):** Explains an axis (like t-SNE 1) using a **combination** of features via models like LASSO or Random Forest. This is the most accurate way to describe "deep" phenotypes.

The interactive beginner tutorial also lets you test component–property correlations easily with sliders ([01_interactive_quickstart.ipynb](01_interactive_quickstart.ipynb)).

### 7.1 Global explainers: embedding dimensions vs properties

This step asks which classical properties align with the model’s embedding **across dimensions** (dataset-wide). Use `property_keys=None` to use all computed properties; here we use the first 20 keys to keep the notebook fast.

In [ ]:
subset_keys = pheno.get_available_property_keys()[:20]

emb_corr = pheno.compute_embedding_property_correlations(property_keys=subset_keys)
summary_df = pheno.summarize_embedding_property_correlations(
    correlations=emb_corr,
    order_by="mean_abs",
    top_k=15,
    plot=True,
)

### 7.2 Component explainers: map axes vs properties

Ties each **dimensionality-reduction axis** (PCA, t-SNE, or UMAP) to classical properties. For the same story as a published figure, use the same `method` and optional `filters` as that plot (e.g. `method="tsne"`, `filters={"condition": [...]}`). Below we use **PCA** for a minimal example.

In [ ]:
corr_comp = pheno.compute_component_correlation(
    method="pca",
    n_components=2,
    top_k=5,
)

### 7.3 Multivariate Explainability (LASSO)

Classical correlation (above) only looks at one feature at a time. However, deep learning models rarely look at just "Area" or "Circularity" — they use highly complex, entangled combinations of features to group cells together. **Multivariate interpretability** addresses this by training a regression model (like LASSO) to predict an entire abstract embedding axis (like t-SNE Component 1) using *all* classical features simultaneously.

**Why do this? (The Motivation)**
It allows you to "open the black box" and explicitly translate an abstract deep learning visualization into a readable, geometric formula. 

**How to interpret the output:**
1. **Explainability Score ($R^2$):** Tells you how much of the deep learning model's logic can be explained by standard biology. 
   *   **High $R^2$ (e.g., 0.85):** Excellent. 85% of what the AI is using to separate clusters along this axis is purely geometric (like shape + size).
   *   **Low $R^2$ (e.g., 0.15):** Fascinating. The AI has discovered a complex sub-resolution texture or novel phenotype that standard classical properties are completely blind to.
2. **Top Drivers (Weights):** Shows you the exact *combination* of features acting together. LASSO is intentionally sparse—it ignores irrelevant features and only shows you the most important ones acting simultaneously (e.g., `[+0.45] Eccentricity + [-0.30] Intensity Entropy`).

In [ ]:
# Explain t-SNE Component 1 using a combination of classical properties.
# By default this also shows an interactive driver bar chart (plot=True).
# Use plot=False to log a plain-text table via the phenome logger instead.
lasso_results = pheno.compute_multivariate_interpretability(
    method="tsne",
    component=1,
    model_type="lasso",
    top_k=10,
)

## 8. Interactive Exploration

The pipeline provides an interactive dashboard using Plotly and ipywidgets. This allows you to explore the embedding space, highligh specific groups, and inspect individual images by clicking on dots.

Note: This requires a Jupyter environment with `ipywidgets` and `plotly` installed.


In [ ]:
# Launch interactive explorer (if in Jupyter)
explorer = pheno.create_interactive_explorer()

## See Also

- [03_core_phenotyping_workflow.ipynb](03_core_phenotyping_workflow.ipynb): Core API tutorial
- [05_extending_phenome_plugins.ipynb](05_extending_phenome_plugins.ipynb): Writing custom plugins and model wrappers